# Attention is all You Need

- This is the name of the paper that introduced the transformer. Linked [here](https://arxiv.org/abs/1706.03762)
- We will be training the transformer model on the [Tiny Shakespeare dataset](https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt). This is a concatenation of all of shakesepeare's works. The idea is that the transformer model will look at prior characters and will predict the next character in the sequence. The transformer will just try to produce the correct sequences
- Characters are being released on a token-by-token level (tokens are subword pieces)
-nanoGPT is used to train the transformers on the given text. This is composed of two files where one file defines the GPT model and the other trains the model on some given text dataset

In [1]:
from structures.config import get_params
import requests

## Reading and Exploring the Data

In [2]:
# Define a function to get text data from a url
def get_text(url: str) -> str:
    response = requests.get(url)
    if response.status_code == 200:
        byte_data = response.content
        text = byte_data.decode('utf-8')
        return text

In [3]:
params = get_params()
text_data = get_text(params.tiny_shakespeare)

In [4]:
len(text_data) # Number of characters in the text

1115394

In [5]:
# unique characters in the text, sorted
characters = sorted(list(set(text_data))) # Convert to set so that we only have a list of unique characters, then convert to list and sort for ordering
vocab_size = len(characters) # Gives the number of unique characters we have
print(vocab_size) # Prints out the number of characters in our text data
print(''.join(characters)) # Prints out each character that appearas in our text data

65

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


## Tokenization, Train/Val Split

In [6]:
# Creating an encoder and decoder - this translates characters to integers and vice versa

# The below code depicts a simple example of an encoder and decoder. There are many others. Our encoding/decoding works on the character level
## Other encoders and decoders, such as Google's SentencePiece will not use entire words nor individual characters, rather sub-words. This is the standard practice
## OpenAI uses Byte-pair encoding tokenizer called 'tiktoken' 
char_to_int = {character:idx for idx, character in enumerate(characters)}
int_to_char = {idx:character for idx, character in enumerate(characters)}

encoder = lambda s: [char_to_int[character] for character in s] # Encodes the character into a value
decoder = lambda l: ''.join([int_to_char[integer] for integer in l]) # Decodes the values to get the characters

print(encoder('Hello, World!')) # Gives back some list of numbers
print(decoder(encoder('Hello, World!'))) # Should give back 'Hello, World!'

[20, 43, 50, 50, 53, 6, 1, 35, 53, 56, 50, 42, 2]
Hello, World!


In [7]:
# Example using tiktoken
import tiktoken

enc = tiktoken.get_encoding('gpt2') # Getting the GPT-2 encoder
print(enc.n_vocab) # Gives the number of possible tokens in gpt-2's encoder. GPT-2 has tokens with values from 0 to 50256!

# You can trade off between code book size and sequence lengths! 
## This means you can have very long sequences of integers with very small vocabularies or very large sequences of integers with very small vocabularies
## The encoder/decoder we made earlier is very simple. We have very simple encode and decode functions but very large sequences of integers as a result

encoded_tiktoken = enc.encode('Hello, World!')
print(encoded_tiktoken)
decoded_tiktoken = enc.decode(encoded_tiktoken)
print(decoded_tiktoken)

50257
[15496, 11, 2159, 0]
Hello, World!


In [8]:
# We can now tokenize the entire training set of shakespeare
import torch
data = torch.tensor(encoder(text_data), dtype=torch.long) #wrapping our values in a pytorch tensor
print(data.shape, data.dtype)
print(data[:1000]) #This is the first 1000 characters being fed into GPT

c:\Users\omara\Desktop\workspace\gpt_from_scratch\venvironment\Lib\site-packages\torch\nn\modules\transformer.py:20: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at ..\torch\csrc\utils\tensor_numpy.cpp:84.)
  device: torch.device = torch.device(torch._C._get_default_device()),  # torch.device('cpu'),


torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [9]:
# Separate our data into train and validation (90-10 split)
n = int(.9 * len(data))
train_data = data[:n]
val_data = data[n:] # We withhold some amount of data to keep the neural net from memorizing the data. That's because we effectively want to be able to generate shakespeareian text!

## Data Loader: Batches of chunks of data

We don't want to feed the data in all at once to the transformer, as it's very computationally expensive. Instead we feed it training data in random chunks.  
There's usually a maximum length to the chunk size of the data! This is usually denoted as "block size"

In [10]:
block_size = 8
train_data[:block_size+1] # Gives the first 9 characters of the training set

## IMPORTANT NOTE: It is important to understand the following - We are predicting sequences of characters. 
### Therefore, whatever chunk size of training data we have, the corresponding predicted value is character that follows that chunk. 
### For example: if we have the following chunk of characters [18, 47, 56, 57], then the value we are predicting from the first 3 characters is the 4th character! 
### So for a chunk of 9 characters we have 8 predicters!

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [11]:
x = train_data[:block_size]
y = train_data[1:block_size+1] # Offsetting everything by 1
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

# The below output is the chunk of 8 examples hidden in the 9 characters!
## This is done to make the transformer accustomed to seeing context from as little as 9 character all the way up to the block size. We want the transformer to be used to seeing everything in between
## This is going to be useful later during inference because while we sample we can start the sampling duration with as little as one character of context and then it can start predicting from one character
## all the way up to the block size.
## After the block size we start truncating because the transformer will never receive more than the block size in inputs when it's predicting its next character.
## This all encompasses the time dimension of a transformer model

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [12]:
# The batch dimension
## As we're sampling these chunks of text, every time we're going to be feeding them into a transformer we're going to have many batches of multiple chunks of text that are all stacked in a single tensor
## This is done for efficiency because of their parallel processing capabilities
## We process multiple chunks at the same time. They're processed independently


torch.manual_seed(1337) # Sets the seed for generating random numbers
batch_size = 4 # How many independent sequences we'll run in parallel. This is how many we're processing on every forward and backward pass of the transformer
block_size = 8 # What is the maximum context length for predictions

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data 

    # The second argument tells us how many random offsets are being produced. So since batch_size is 4, ix is 4 randomly generated numbers between 0 and the length of the data minus the block size
    ix = torch.randint(len(data) - block_size, (batch_size,)) 

    # These take all the 1-D tensors we had and stacks them as rows. Therefore they all become a row in a 4 by 8 tensors
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix]) # Recall that our y values are the offset by 1 of the x values. These will be used by the transformer to compute the loss function when we generate values for the inputs
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('outputs:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time (sequence) dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")


## Consider how powerful this is - the input 4x8 array contains a total of 32 examples! Furthermore, they're all independent as far as the transformer is concerned

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
outputs:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

## Simplest Baseline: Bigram Language Model, Loss, Generation

According to Andrej Karpathy, this is the simplest possible neural network in language models

Brief Aside: Embeddings
- Token embeddings are a way of representing words or tokens as dense vectors in a continuous vector space
- The embeddings capture semantic relationships between tokens, allowing models to learn contextual information about words
- In NLP, token embedding tables are used in NN models. The tables are essentially a matrix where each row corresponds to the embedding of a specific token
- The number of embeddings of a token are like the dimensionality of our feature space. They allow us to determine semantic similarity and relationships between words. For example, the words "King" and "Queen" will have similar embedding vectors because of their semantic similarity. The higher the number of embeddings for the dimensionality, the more likely you are to overfit

In [13]:
import torch
import torch.nn as nn 
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module): # This model is a subclass of nn.Module!
    def __init__(self, vocab_size):
        super().__init__()
        # Each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    # Evaluates the quality of the model
    def forward(self, idx, targets=None):
        # idx and targets are both (B, T) tensor of integers - they are both Batch by Time
        logits = self.token_embedding_table(idx) # (B,T,C) - Batch by Time by Channel. Channel is our vocab size in this case, which is 65
        
        if targets is None:
            loss = None 
        else:
            # Playing around with dimensions for Pytorch's syntax purposes. Cross entropy wants (B, C, T), which is a pain so instead we'll do this
            B, T, C = logits.shape
            # view() function is how we reshape pytorch tensors. 
            # Note that we're multiplying B*T because that will yield the new length we need!
            logits = logits.view(B*T, C) # Stretching out our Batch and Time dimensions into one vector and then keep the Channel dimension as the second dimension
            targets = targets.view(B*T) # Strecthing out our Batches and Targets into a one dimensional vector
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    # Generation function of the model
    def generate(self, idx, max_new_tokens):
        # idx is (B,T) array of indices in the current context
        # The job of this function is to take the (B,T) and extend it by +1, +2, +3, etc. for each batch dimension in the time dimension. It'll continue this for max_new_tokens
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # Becomes (B, C). We only want the very last set of logits since that's what is relevent with each iteration of the loop
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=1) # (B, C)
            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1) - this basically just returns the argmax (highest probability character)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1) # This part is where we concatenate onto the time dimension
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb) # Passing in the inputs and targets
print(logits.shape)
print(loss)
idx = torch.zeros((1,1), dtype=torch.long) # This is a 1 by 1 tensor containing zero that we use to kick things off. This is a good value since 0 decodes to a \n character
print(decoder(m.generate(idx, max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

Sr?qP-QWktXoL&jLDJgOLVz'RIoDqHdhsV&vLLxatjscMpwLERSPyao.qfzs$Ys$zF-w,;eEkzxjgCKFChs!iWW.ObzDnxA Ms$3


## Training the Bigram Model